# VCF Coordinate Remap + Flanking-Sequence Sanity Check

**Part A** — Use `rice_genome_query` utilities to produce a new VCF whose positions are in MSU7/IRGSP1 coordinates, so it aligns directly with `GCA_rice.fasta`.

**Part B** — Reproduce the flanking-sequence sanity check from `data_investigation.ipynb`, but using the remapped VCF + plain pysam/pyfaidx calls (no manual coordinate arithmetic).

In [ ]:
import pandas as pd
import pysam
from pyfaidx import Fasta

crop_embed.data.coords import (
    FASTA_PATH,
    VCF_PATH,
    FLANKING_PATH,
    build_msu6_to_msu7_map,
    remap_vcf_coordinates,
)

REMAPPED_VCF_PATH = "../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413_msu7.vcf"

## Part A — Generate the MSU7-coordinate VCF

In [ ]:
coord_map = build_msu6_to_msu7_map()
print(f"Coordinate map covers {len(coord_map):,} SNPs")
coord_map.head()

In [ ]:
counts = remap_vcf_coordinates(VCF_PATH, REMAPPED_VCF_PATH, coord_map)
counts

In [ ]:
# Diagnose the mismatch — inspect raw VCF records vs coord_map keys
vcf_debug = pysam.VariantFile(VCF_PATH)

print("First 5 VCF records:")
print(f"  {'chrom':>10}  {'rec.pos (0-based)':>18}  {'rec.pos+1':>10}  {'id':>12}")
for i, rec in enumerate(vcf_debug.fetch()):
    if i >= 5:
        break
    print(f"  {rec.chrom!r:>10}  {rec.pos:>18}  {rec.pos+1:>10}  {str(rec.id):>12}")
vcf_debug.close()

print("\nFirst 5 coord_map entries (index = (chr, pos_msu6), value = pos_msu7):")
print(coord_map.head())

print("\nIndex dtypes:", coord_map.index.dtypes.tolist())

# Check the first VCF record manually
vcf_debug = pysam.VariantFile(VCF_PATH)
rec = next(vcf_debug.fetch())
vcf_debug.close()

chrom_str = rec.chrom
chrom_int = int(chrom_str) if chrom_str.isdigit() else int(chrom_str.lstrip("chr"))
pos_1based = rec.pos + 1
key = (chrom_int, pos_1based)
print(f"\nFirst record key: {key}  →  coord_map.get: {coord_map.get(key)}")
print(f"Key types: {type(chrom_int)}, {type(pos_1based)}")
print(f"Index level types: {type(coord_map.index[0][0])}, {type(coord_map.index[0][1])}")

In [ ]:
# Inspect the 1,211 dropped records
flanking_by_id  = pd.read_csv(FLANKING_PATH, sep="\t").set_index("snp_id")
flanking_by_pos = pd.read_csv(FLANKING_PATH, sep="\t").set_index(["chr", "pos"])

dropped = []
vcf_debug = pysam.VariantFile(VCF_PATH)
for rec in vcf_debug.fetch():
    chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    if coord_map.get((chrom_int, rec.pos)) is None:
        dropped.append({
            "snp_id":  rec.id,
            "chrom":   chrom_int,
            "pos_vcf": rec.pos,
            "in_flanking_by_id":  rec.id in flanking_by_id.index,
            "flanking_pos":       flanking_by_id.loc[rec.id, "pos"] if rec.id in flanking_by_id.index else None,
        })
vcf_debug.close()

dropped_df = pd.DataFrame(dropped)
print(f"Total dropped: {len(dropped_df)}")
print(f"\nID exists in flanking_seq (position mismatch): {dropped_df['in_flanking_by_id'].sum()}")
print(f"ID absent from flanking_seq entirely:          {(~dropped_df['in_flanking_by_id']).sum()}")

# For position mismatches, show how far off the positions are
mismatches = dropped_df[dropped_df["in_flanking_by_id"]].copy()
if len(mismatches):
    mismatches["pos_delta"] = mismatches["flanking_pos"] - mismatches["pos_vcf"]
    print("\nPosition delta distribution (flanking_pos - pos_vcf):")
    print(mismatches["pos_delta"].value_counts().head(10))

dropped_df.head(10)

## Part B — Flanking-sequence sanity check

For each SNP in the remapped VCF, query the reference FASTA at `rec.pos` (now directly in MSU7 / 0-based coordinates) and compare the ±16 nt flanks against the stored `X5p_MSU6` / `X3p_MSU6` values.

A high match rate confirms the remap is correct.

In [ ]:
# Load the stored flanking sequences, keyed by snp_id
flanking_seq = pd.read_csv(FLANKING_PATH, sep="\t").set_index("snp_id")

ref        = Fasta(FASTA_PATH)
chrom_name = {i + 1: name for i, name in enumerate(ref.keys())}
print("Chromosomes in FASTA:", chrom_name)

In [ ]:
HALF_WINDOW = 16

results = []   # list of dicts: snp_id, chr, pos_msu7, left_match, right_match

vcf = pysam.VariantFile(REMAPPED_VCF_PATH)
for rec in vcf.fetch():
    snp_id = rec.id
    if snp_id not in flanking_seq.index:
        continue

    chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    seq_key   = chrom_name[chrom_int]

    # rec.pos is 0-based MSU7 — slice directly, no offset needed
    start  = max(0, rec.pos - HALF_WINDOW)
    window = ref[seq_key][start : rec.pos + HALF_WINDOW + 1].seq.upper()

    if len(window) != 2 * HALF_WINDOW + 1:
        continue   # near chromosome boundary

    stored = flanking_seq.loc[snp_id]
    results.append({
        "snp_id":      snp_id,
        "chr":         chrom_int,
        "pos_msu7":    rec.pos + 1,   # 1-based for display
        "left_match":  window[:HALF_WINDOW]  == stored["X5p_MSU6"].upper(),
        "right_match": window[HALF_WINDOW+1:] == stored["X3p_MSU6"].upper(),
    })

vcf.close()
results_df = pd.DataFrame(results)
print(f"SNPs checked: {len(results_df):,}")

In [ ]:
both_match = results_df["left_match"] & results_df["right_match"]
n          = len(results_df)

print(f"Both flanks match : {both_match.sum():,} / {n:,}  ({100 * both_match.mean():.2f}%)")
print(f"Left flank only   : {(results_df['left_match'] & ~results_df['right_match']).sum():,}")
print(f"Right flank only  : {(~results_df['left_match'] & results_df['right_match']).sum():,}")
print(f"Neither           : {(~results_df['left_match'] & ~results_df['right_match']).sum():,}")

In [ ]:
# Per-chromosome breakdown
results_df.groupby("chr").apply(
    lambda g: pd.Series({
        "n_snps":       len(g),
        "both_match":   (g["left_match"] & g["right_match"]).sum(),
        "pct_match":    f"{100 * (g['left_match'] & g['right_match']).mean():.1f}%",
    }),
    include_groups=False,
)